# Упражнение: сколько велосипедов возьмут в прокат

**Дисциплина «ИИ и машинное обучение»** · НИУ ВШЭ, Высшая школа бизнеса

Магистратура «Бизнес-аналитика и цифровые решения в финансах»

---

Городской велопрокат каждый вечер решает, сколько велосипедов подвезти
к стойкам на завтра и сколько людей вывести в смену. Решение опирается
на прогноз: сколько поездок будет в каждый час. Здесь вы строите
такой прогноз сами — на 8 760 часах работы проката Сеула за год.

Главное в этой задаче — прогноз делается **на будущее**: модель учится
на прошедших месяцах, а работать ей предстоит на тех, которых она ещё
не видела. Это меняет и то, как делить данные на части, и то, каким
признакам можно доверять.

| Часть | О чём |
|---|---|
| 1 | Задача, данные и деление выборки по времени, а не случайно |
| 2 | Лестница моделей: от температуры до произведений погодных признаков |
| 3 | Проверка на будущем месяце и запас, который закладывает прокат |

**Как работать.** Ячейки с заголовком **Задание** содержат `...` или
`raise NotImplementedError` — это место надо заменить своим кодом.
Самопроверка стоит в конце той же ячейки: если всё верно, она печатает
`✓ верно`. Ячейки выполняются сверху вниз: каждая опирается на результат
предыдущей.

In [ ]:
## ── библиотеки ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# конвейер: последовательность «подготовка данных → модель»
from sklearn.pipeline import Pipeline
# своя обработка для каждой группы столбцов
from sklearn.compose import ColumnTransformer
# масштабирование чисел, кодирование категорий, попарные произведения
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
# сами модели: обычная линейная регрессия и она же со штрафом за большие веса
from sklearn.linear_model import LinearRegression, Ridge
# случайное деление выборки — понадобится один раз, для сравнения
from sklearn.model_selection import train_test_split
# копия необученной модели: чтобы новая модель не переучивала старую
from sklearn.base import clone
# метрики: средняя ошибка в велосипедах и доля объяснённого разброса
from sklearn.metrics import mean_absolute_error, r2_score

## ── настройки печати и графиков ────────────────────────────────────
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 120)
plt.rcParams.update({
    "figure.figsize": (7.2, 3.8), "figure.dpi": 110, "font.size": 11,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})
BLUE, INK, RED, GREY, GREEN = "#0057b8", "#0d2340", "#c0392b", "#8a9bb0", "#1b7f5a"
SEED = 42

## ── вспомогательные функции ────────────────────────────────────────
LADDER = []


def check(ok, hint):
    "Самопроверка задания."
    if not ok:
        raise AssertionError("✗ " + hint)
    print("✓ верно")


def score(tag, y_true, pred, remember=True):
    "Печатает качество прогноза и запоминает шаг для итоговой лестницы."
    mae = mean_absolute_error(y_true, pred)
    r2 = r2_score(y_true, pred)
    if remember:
        LADDER.append((tag, mae))
    print("%s: MAE %.1f велосипедов за час, R² %.3f" % (tag, mae, r2))
    return mae


print("Библиотеки загружены.")

---
# Часть 1. Задача и данные

## Шаг 0. Что мы решаем

Вечером диспетчер проката смотрит на завтрашний план и решает две вещи:
сколько велосипедов развезти по стойкам и сколько курьеров вывести
в смену. Обе ошибки стоят денег, но по-разному:

* **занизили спрос** — к вечеру стойки в центре пусты, поездки не
  состоялись, клиенты ушли к конкуренту;
* **завысили спрос** — велосипеды и люди простаивают, развозка съедает
  деньги, которые никто не вернёт.

Мерить будем **MAE — среднюю ошибку в велосипедах за час**: это те самые
штуки, которые диспетчер грузит в машину. Рядом смотрим R² — долю
разброса спроса, которую модель объяснила.

Данные — час за часом за целый год работы проката Сеула: спрос, погода,
календарь. Набор открытый (Seoul Bike Sharing Demand, UCI,
[archive.ics.uci.edu/dataset/560](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand)),
в репозитории курса лежит его готовая версия: названия столбцов и
значения переведены на русский, видимость переведена из десятков метров
в метры. **Больше ничего не делали** — всё остальное осталось как есть.

In [ ]:
## ── данные ─────────────────────────────────────────────────────────
# read_csv читает таблицу прямо по ссылке: скачивать ничего не нужно
BIKES = ("https://raw.githubusercontent.com/Rimest/ml_ai_course_gsb_hse"
         "/main/data/seoul_bikes.csv")
bikes = pd.read_csv(BIKES, parse_dates=["дата"])

print("Строк: %d, столбцов: %d" % bikes.shape)

bikes.head()

Одна строка — один час работы проката. Столбцы:

| Столбец | Что в нём |
|---|---|
| `дата`, `час` | когда: дата и час суток от 0 до 23 |
| `аренды` | сколько велосипедов взяли за этот час — **это ответ** |
| `температура`, `влажность`, `ветер` | °C, %, м/с |
| `видимость` | метры |
| `точка_росы` | °C, температура, при которой выпадает роса |
| `солнце` | солнечное излучение, МДж/м² |
| `дождь`, `снег` | мм и см за час |
| `сезон`, `праздник` | зима, весна, лето, осень; да или нет |
| `прокат_работал` | работал ли прокат в этот час |

### Задание 1.1
Посчитайте, за какой период собраны данные, каков средний и медианный
спрос за час и сколько часов спрос оказался ровно нулевым.

In [ ]:
## ── осмотр ─────────────────────────────────────────────────────────
пустых_часов = ...  # TODO: сколько строк, где «аренды» равны нулю

## ── числа ──────────────────────────────────────────────────────────
print("Период: с %s по %s (%d дней)" % (
    bikes["дата"].min().date(), bikes["дата"].max().date(),
    bikes["дата"].nunique()))
print("Спрос за час: в среднем %.0f, медиана %.0f, максимум %d"
      % (bikes["аренды"].mean(), bikes["аренды"].median(), bikes["аренды"].max()))
print("Часов с нулевым спросом: %d" % пустых_часов)
print("Пропусков в таблице: %d" % bikes.isna().sum().sum())

## ── самопроверка ───────────────────────────────────────────────────
check(пустых_часов == 295, "нужно число строк, а не строки целиком")

Ровно ноль поездок за час — для города с миллионом жителей странно даже
глубокой ночью: в четыре утра прокат берут около 130 раз. Значит, эти
295 часов чем-то отличаются от остальных. В таблице есть столбец, который
это объясняет.

### Задание 1.2
Посмотрите, как нулевые часы распределены по столбцу `прокат_работал`,
и оставьте в таблице `work` только те часы, когда прокат работал.

In [ ]:
## ── откуда нули ────────────────────────────────────────────────────
print("Средний спрос по столбцу «прокат_работал»:")
print(bikes.groupby("прокат_работал")["аренды"].agg(["size", "mean"]).to_string())

## ── оставляем только рабочие часы ──────────────────────────────────
work = ...  # TODO: строки, где «прокат_работал» равен «да», обязательно с .copy()

## ── числа ──────────────────────────────────────────────────────────
print("")
print("Часов осталось: %d из %d" % (len(work), len(bikes)))
print("Спрос за рабочий час: в среднем %.0f" % work["аренды"].mean())

## ── самопроверка ───────────────────────────────────────────────────
check(len(work) == 8465, "должно остаться 8 465 часов")
check(work["аренды"].min() > 0, "в оставшихся часах нулевого спроса быть не должно")

В закрытые часы спрос не нулевой — его просто нет: дверь закрыта,
и взять велосипед невозможно. Если оставить эти строки, модель будет
учиться предсказывать закрытие проката по погоде, а на проверке получит
штраф за то, чего предсказать не могла. Такие строки убирают из всех
частей выборки сразу — и из обучающей, и из проверочных.

## Шаг 1. Как делить выборку, когда прогноз нужен на будущее

Обычно выборку делят случайно: перемешали строки и отрезали части.
Здесь так делать нельзя, и причина простая.

Диспетчер планирует **завтрашний** день, имея на руках **вчерашние**
данные. Значит, и проверять модель надо так же: учить на прошлом,
проверять на будущем. Если перемешать часы случайно, в обучающую часть
попадут соседние часы того же самого дня, что и в проверочную, — модель
будет знать про этот день почти всё, а на настоящем прогнозе такой
роскоши не будет.

Поделим по календарю:

* **обучение** — с декабря 2017 по август 2018 включительно;
* **валидация** — сентябрь и октябрь 2018, на ней сравниваем модели;
* **тест** — ноябрь 2018, его открываем один раз в конце.

### Задание 1.3
Разделите `work` на три части по датам. Границы: `"2018-09-01"`
и `"2018-11-01"`. Не забудьте `.copy()` — дальше мы будем добавлять
в части новые столбцы.

In [ ]:
## ── делим по времени ───────────────────────────────────────────────
# TODO: три среза по столбцу «дата» с границами 2018-09-01 и 2018-11-01
raise NotImplementedError("замените эту строку своим решением")

## ── числа ──────────────────────────────────────────────────────────
print("Части:")
for name, part in (("обучение", train), ("валидация", valid), ("тест", test)):
    print("   %-11s %5d часов, с %s по %s, средний спрос %4.0f"
          % (name, len(part), part["дата"].min().date(),
             part["дата"].max().date(), part["аренды"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(len(train) == 6528 and len(valid) == 1289 and len(test) == 648,
      "должно получиться 6 528 / 1 289 / 648 часов")
check(train["дата"].max() < valid["дата"].min() < test["дата"].min(),
      "части должны идти во времени одна за другой, без пересечений")

Части получились непохожими, и это нормально: в обучении средний спрос
671 велосипед за час, в валидации — 1 027. Сентябрь и октябрь в Сеуле
тёплые и сухие, и прокат в них работает заметно активнее, чем в среднем
за год. Модели придётся дотянуться до этого уровня по погоде: она не
видела ни одного осеннего часа.

Насколько случайное деление было бы мягче, видно сразу — посчитаем,
сколько часов случайной проверочной части имели бы соседний час того же
дня в обучении.

In [ ]:
## ── что было бы при случайном делении ──────────────────────────────
случ_обучение, случ_проверка = train_test_split(
    work, test_size=0.2, random_state=SEED)
случ_обучение, случ_проверка = случ_обучение.copy(), случ_проверка.copy()

есть_сосед = set(zip(случ_обучение["дата"], случ_обучение["час"]))
рядом = sum(((д, ч - 1) in есть_сосед) or ((д, ч + 1) in есть_сосед)
            for д, ч in zip(случ_проверка["дата"], случ_проверка["час"]))

## ── числа ──────────────────────────────────────────────────────────
print("Часов в случайной проверочной части: %d" % len(случ_проверка))
print("из них соседний час того же дня попал в обучение: %d (%.0f %%)"
      % (рядом, 100 * рядом / len(случ_проверка)))

95 % проверочных часов имеют соседа в обучении. Спрос в 19:00 почти
всегда похож на спрос в 18:00 того же дня: та же погода, тот же день
недели, те же люди. Модель, проверенная так, показывает красивое число,
но отвечает на вопрос «угадаешь ли ты пропущенный час посреди известного
дня» — а диспетчеру нужен ответ на другой вопрос. В конце мы сравним оба
числа.

## Шаг 2. Календарь: будни и выходные

Один и тот же час значит разное в понедельник и в воскресенье. Сделаем
признак, который это различает: выходным считаем субботу, воскресенье
и праздник.

Метод `.dt.dayofweek` даёт номер дня недели: 0 — понедельник, 6 —
воскресенье. `np.where(условие, "да", "нет")` собирает столбец из двух
значений по условию.

### Задание 1.4
Добавьте в `train`, `valid` и `test` столбец `выходной` со значениями
`"да"` и `"нет"`: «да» — если суббота, воскресенье или праздник.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
for part in (train, valid, test):
    # TODO: np.where по двум условиям: день недели 5 или 6, либо праздник «да»
    raise NotImplementedError("замените эту строку своим решением")

## ── числа ──────────────────────────────────────────────────────────
print("Часов, помеченных выходными:")
for name, part in (("обучение", train), ("валидация", valid), ("тест", test)):
    print("   %-11s %4d из %4d" % (name, (part["выходной"] == "да").sum(), len(part)))
print("")
print("Средний спрос: будни %.0f, выходные %.0f"
      % (train.loc[train["выходной"] == "нет", "аренды"].mean(),
         train.loc[train["выходной"] == "да", "аренды"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(set(train["выходной"]) == {"да", "нет"},
      "в столбце должны быть только «да» и «нет»")
check((train["выходной"] == "да").sum() == 2112,
      "в обучении должно получиться 2 112 выходных часов")

В среднем за час разница небольшая — но среднее здесь скрывает главное.
Посмотрим на спрос по часам суток отдельно для будней и выходных.

### Задание 1.5
Постройте сводную таблицу: строки — часы суток, столбцы — `выходной`,
значения — средний спрос. Пригодится `pivot_table`.

In [ ]:
## ── профиль дня ────────────────────────────────────────────────────
профиль = ...  # TODO: pivot_table по train: index «час», columns «выходной», values «аренды»

## ── график ─────────────────────────────────────────────────────────
fig, ax = plt.subplots()
ax.plot(профиль.index, профиль["нет"], color=BLUE, marker="o", ms=3, label="будни")
ax.plot(профиль.index, профиль["да"], color=RED, marker="o", ms=3, label="выходные")
ax.set_xlabel("час суток")
ax.set_ylabel("средний спрос, велосипедов")
ax.set_title("Будни и выходные устроены по-разному")
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout()
plt.show()

## ── самопроверка ───────────────────────────────────────────────────
check(профиль.shape == (24, 2), "в таблице должно быть 24 строки и 2 столбца")
check(профиль["нет"].idxmax() == 18, "пик буднего дня приходится на 18 часов")

Два разных дня. В будни спрос двугорбый: утренний пик в 8 часов и
вечерний, вдвое выше, в 18 — люди едут на работу и с работы. В выходные
горб один и пологий: спрос плавно растёт с утра до пяти вечера. В восемь
утра разница трёхкратная: 1 236 велосипедов в будни против 390
в выходные.

Запомним эту картинку: она объясняет половину того, что будет дальше.

---
# Часть 2. Лестница моделей

Дальше всё устроено одинаково: строим модель, меряем MAE **на
валидации**, записываем результат и думаем, чего модели не хватает.
Первый ориентир — сколько ошибётся тот, кто вообще не строит моделей
и на каждый час называет одно и то же число.

In [ ]:
## ── ориентир: одно число на все часы ───────────────────────────────
guess = np.full(len(valid), train["аренды"].mean())

score("0. всем среднее обучения", valid["аренды"], guess)

593 велосипеда ошибки при среднем спросе около тысячи — вот что надо
обыграть.

## Шаг 3. Первая модель: температура

Самая очевидная мысль: в тепло катаются, в мороз нет. Соберём конвейер
из двух шагов и обучим его на одном признаке.

* `StandardScaler()` — приводит числовые столбцы к общему масштабу
  (вычитает среднее, делит на разброс), чтобы веса модели были
  сравнимы между собой;
* `LinearRegression()` — подбирает веса так, чтобы сумма квадратов
  промахов была наименьшей;
* `Pipeline([...])` — склеивает шаги в одну модель: `fit` применяет их
  по очереди, `predict` — тоже, и подготовка не может «подсмотреть»
  проверочную часть.

Спрос не бывает отрицательным, а линейная модель об этом не знает,
поэтому прогноз обрезаем снизу нулём: `np.clip(pred, 0, None)`.

### Задание 2.1
Обучите конвейер на столбце `температура` и замерьте качество на
валидации.

In [ ]:
## ── модель ─────────────────────────────────────────────────────────
ЧИСЛА = ["температура"]

model = Pipeline([
    ("масштаб", StandardScaler()),
    ("модель", LinearRegression()),
])
# TODO: обучите model на train: признаки — train[ЧИСЛА], ответ — train["аренды"]
raise NotImplementedError("замените эту строку своим решением")

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid[ЧИСЛА]), 0, None)
mae = score("1. температура", valid["аренды"], pred)

## ── самопроверка ───────────────────────────────────────────────────
check(mae < 560, "ошибка должна стать заметно меньше, чем у среднего")

514 против 593. Температура работает, но объясняет мало: R² почти нулевой
— модель едва лучше одной цифры на все часы.

Посмотрим, как спрос устроен по температуре, прежде чем добавлять
признаки.

In [ ]:
## ── спрос по температуре ───────────────────────────────────────────
корзины = pd.cut(train["температура"], [-20, -5, 0, 5, 10, 15, 20, 25, 30, 40])
по_температуре = train.groupby(корзины)["аренды"].mean()

fig, ax = plt.subplots()
ax.bar(range(len(по_температуре)), по_температуре.values, color=BLUE)
ax.set_xticks(range(len(по_температуре)))
ax.set_xticklabels(["-20…-5", "-5…0", "0…5", "5…10", "10…15",
                    "15…20", "20…25", "25…30", "30+"], fontsize=9)
ax.set_xlabel("температура, °C")
ax.set_ylabel("средний спрос, велосипедов")
ax.set_title("Спрос растёт с теплом, но не бесконечно")
plt.tight_layout()
plt.show()

Связь есть, но она не прямая: спрос растёт до 25–30 °C, а в жару за
тридцать снова падает. Прямая линия так не умеет — она обязана либо
недооценить тепло, либо переоценить жару. Мы вернёмся к этому в конце
части.

## Шаг 4. Час суток

Спрос в 4 утра и в 18 часов различается в десять раз, а модель об этом
не знает. Добавим час — сначала самым простым способом, числом от 0 до 23.

### Задание 2.2
Добавьте `час` в список числовых признаков и переобучите ту же модель.

In [ ]:
## ── модель ─────────────────────────────────────────────────────────
ЧИСЛА = ...  # TODO: список из двух столбцов: температура и час

model = Pipeline([
    ("масштаб", StandardScaler()),
    ("модель", LinearRegression()),
])
model.fit(train[ЧИСЛА], train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid[ЧИСЛА]), 0, None)
mae = score("2. + час числом", valid["аренды"], pred)

## ── что модель выучила про час ─────────────────────────────────────
вес_часа = model.named_steps["модель"].coef_[1] / train["час"].std()
print("Вес часа: %+.0f велосипеда на каждый час вперёд" % вес_часа)

## ── самопроверка ───────────────────────────────────────────────────
check(len(ЧИСЛА) == 2 and "час" in ЧИСЛА, "в списке должны быть температура и час")
check(mae < 470, "ошибка должна опуститься примерно до 430")

Ошибка упала до 431, но посмотрите, что модель выучила: «каждый
следующий час прибавляет к спросу около 31 велосипеда». То есть в
полночь спрос должен быть максимальным, а в 4 утра — почти нулевым.
На графике профиля дня видно, что это неправда: спрос дважды растёт
и дважды падает.

Числом час подавать нельзя: у линейной модели один вес на признак,
а значит, только одно направление — вверх или вниз. Нужен другой способ.

## Шаг 5. Час как категория

Отдадим модели час не числом, а меткой: 24 столбца по числу часов,
в каждой строке единица стоит в своём столбце. Тогда у модели будет
своя поправка на каждый час, и форма профиля ей не помешает.

* `OneHotEncoder(handle_unknown="ignore")` — превращает столбец с
  метками в столбцы из нулей и единиц; `handle_unknown="ignore"` велит
  не падать, а ставить нули, если на проверке встретится метка,
  которой не было в обучении;
* `ColumnTransformer([...])` — назначает свою обработку каждой группе
  столбцов: числа масштабируем, категории кодируем.

Обратите внимание: теперь в `fit` и `predict` уходит вся таблица, а не
выборка столбцов, — какие столбцы брать, знает `ColumnTransformer`.

### Задание 2.3
Соберите `ColumnTransformer`, где `температура` идёт через
`StandardScaler`, а `час` — через `OneHotEncoder`, и обучите модель.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
ЧИСЛА = ["температура"]
КАТЕГОРИИ = ["час"]

# TODO: ColumnTransformer из двух частей: StandardScaler на ЧИСЛА, OneHotEncoder на КАТЕГОРИИ
raise NotImplementedError("замените эту строку своим решением")

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])
model.fit(train, train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid), 0, None)
mae = score("3. час категорией", valid["аренды"], pred)

## ── числа ──────────────────────────────────────────────────────────
print("Столбцов после подготовки: %d" % подготовка.transform(train.head(1)).shape[1])

## ── самопроверка ───────────────────────────────────────────────────
check(mae < 410, "ошибка должна опуститься примерно до 390")

390 против 431. Тот же признак, поданный иначе, дал больше сорока
велосипедов — а новых данных мы не добавили ни строчки.

## Шаг 6. Погода целиком

В таблице есть ещё шесть погодных столбцов. Дождь и снег объясняются
сами собой, влажность и видимость — через самочувствие и безопасность,
солнце — через желание выйти на улицу.

### Задание 2.4
Добавьте к температуре остальные погодные столбцы: `влажность`, `дождь`,
`снег`, `ветер`, `видимость`, `солнце`.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
# TODO: список из семи погодных столбцов, начиная с температуры
raise NotImplementedError("замените эту строку своим решением")
КАТЕГОРИИ = ["час"]

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ПОГОДА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])
model.fit(train, train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid), 0, None)
mae = score("4. + погода", valid["аренды"], pred)

## ── веса погодных признаков ────────────────────────────────────────
веса = model.named_steps["модель"].coef_[:len(ПОГОДА)]
print("Вклад погоды (велосипедов на одно стандартное отклонение):")
for имя, вес in sorted(zip(ПОГОДА, веса), key=lambda p: -abs(p[1])):
    print("   %-12s %+7.0f" % (имя, вес))

## ── самопроверка ───────────────────────────────────────────────────
check(len(ПОГОДА) == 7, "погодных столбцов семь")
check(mae < 375, "ошибка должна опуститься примерно до 357")

357. Дальше идут два шага, которые выглядят разумно, но делают хуже.
Оба стоит проделать самому: именно так и выглядит работа с данными.

## Шаг 7. Ловушка первая: логарифм спроса

Спрос сильно скошен вправо: медиана 504 велосипеда, а максимум 3 556.
Для таких величин есть привычный приём — учить модель не на самом
ответе, а на его логарифме: тогда она мерит промах в процентах, а не
в штуках, и редкие огромные значения перестают тянуть её на себя.
Приём напрашивается, проверим его.

`np.log1p(x)` — это логарифм от `x + 1` (спрос бывает маленьким, и
единица спасает от логарифма нуля), `np.expm1` возвращает обратно.

### Задание 2.5
Обучите ту же модель на `np.log1p(train["аренды"])` и верните прогноз
в велосипеды через `np.expm1`.

In [ ]:
## ── обучение на логарифме ──────────────────────────────────────────
подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ПОГОДА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), ["час"]),
])
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])

# TODO: обучите model на логарифме ответа, а прогноз верните в велосипеды через np.expm1
raise NotImplementedError("замените эту строку своим решением")
pred = np.clip(pred, 0, None)

## ── качество на валидации ──────────────────────────────────────────
mae = score("5. логарифм спроса", valid["аренды"], pred)

## ── куда уехал прогноз ─────────────────────────────────────────────
print("Средний прогноз: %.0f велосипедов, факт: %.0f"
      % (pred.mean(), valid["аренды"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(mae > 400, "здесь ошибка должна вырасти — так и задумано")

434 вместо 357 — стало хуже почти на восемьдесят велосипедов. Почему
разумный с виду приём навредил?

Логарифм заставляет модель мерить ошибку в процентах, а не в штуках.
Промах на 50 велосипедов ночью, когда спрос 130, для неё теперь дороже,
чем промах на 300 в вечерний пик. Модель перестраивается под ночные часы
и в целом занижает прогноз: в среднем она обещает 638 велосипедов при
фактических 1 027.

К этому добавляется второе: в валидации спрос выше всего, что модель
видела в обучении, а логарифм сжимает верх шкалы — дотянуться до высоких
значений он мешает сильнее, чем помогает.

Вывод не в том, что логарифм плох. Вывод в том, что приём работает не
сам по себе, а под метрику и под данные: MAE в штуках и проверка на
периоде с высоким спросом — не тот случай.

## Шаг 8. Ловушка вторая: сезон

В таблице есть готовый столбец `сезон` — казалось бы, самый календарный
признак из возможных. Добавим его категорией.

### Задание 2.6
Добавьте `сезон` в список категорий и переобучите модель (на обычном
спросе, без логарифма).

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
КАТЕГОРИИ = ...  # TODO: список из двух категорий: час и сезон

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ПОГОДА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])
model.fit(train, train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid), 0, None)
mae = score("6. + сезон", valid["аренды"], pred)

## ── какие сезоны где ───────────────────────────────────────────────
print("Сезоны в обучении:  %s" % ", ".join(sorted(train["сезон"].unique())))
print("Сезоны в валидации: %s" % ", ".join(sorted(valid["сезон"].unique())))

## ── самопроверка ───────────────────────────────────────────────────
check(mae > 360, "здесь ошибка тоже должна вырасти")

368 вместо 357. Причина видна в двух последних строчках: обучение
заканчивается августом, поэтому в нём есть зима, весна и лето — но нет
ни одного осеннего часа. А валидация целиком осенняя.

Что происходит внутри. `OneHotEncoder(handle_unknown="ignore")` не падает
на незнакомой метке, а ставит нули во все сезонные столбцы. Но у каждой
обучающей строки один из этих столбцов был единицей, и модель подбирала
веса, рассчитывая на эту поправку. Осенние строки остаются без неё —
прогноз смещается, и ошибка растёт.

Это общее правило для прогноза на будущее: **признак, значения которого
в будущем новые, работать не будет**. Календарные метки — сезон, месяц,
номер недели — почти всегда такие. Погода в этом смысле надёжнее:
двадцать градусов в октябре — это те же двадцать градусов, что были
в мае, и модель знает, что с ними делать.

## Шаг 9. Час и день недели вместе

Вернёмся к профилю дня. Восемь утра в будни — 1 354 велосипеда,
восемь утра в выходные — 422. Модель сейчас держит **одну** поправку
на восемь утра и **одну** на выходной день, а нужна поправка на их
сочетание.

Самый прямой способ — склеить два столбца в один составной признак:
`8_нет`, `8_да`, `9_нет` и так далее. Получится 48 меток вместо 24 + 2.

### Задание 2.7
Добавьте в каждую часть столбец `час_и_день` — час и метку выходного,
склеенные через подчёркивание, — и обучите модель на нём вместо `час`.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
for part in (train, valid, test):
    # TODO: час строкой, подчёркивание и значение столбца «выходной»
    raise NotImplementedError("замените эту строку своим решением")

КАТЕГОРИИ = ["час_и_день"]

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ПОГОДА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])
model.fit(train, train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid), 0, None)
mae = score("7. час вместе с днём недели", valid["аренды"], pred)

## ── числа ──────────────────────────────────────────────────────────
print("Разных меток «час_и_день» в обучении: %d" % train["час_и_день"].nunique())

## ── самопроверка ───────────────────────────────────────────────────
check(train["час_и_день"].nunique() == 48, "должно получиться 48 меток")
check(mae < 345, "ошибка должна опуститься примерно до 336")

336. Мы не добавили ни одного нового измерения — только разрешили модели
смотреть на час и день недели вместе, а не по отдельности.

## Шаг 10. Дождь: важно не сколько, а идёт ли

Столбец `дождь` уже в модели, но он меряет миллиметры, а миллиметров
почти всегда ноль: дождь идёт в 6,6 % часов. Для линейной модели это
означает «прибавь столько-то велосипедов за каждый миллиметр» — а решает
на самом деле сам факт дождя: в дождливые часы спрос падает с 707
до 168 велосипедов.

### Задание 2.8
Добавьте в каждую часть столбец `дождь_идёт` со значениями `"да"` и
`"нет"` и отдайте его модели категорией.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
for part in (train, valid, test):
    # TODO: np.where по условию «дождь больше нуля»
    raise NotImplementedError("замените эту строку своим решением")

КАТЕГОРИИ = ["час_и_день", "дождь_идёт"]

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ПОГОДА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])
model.fit(train, train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid), 0, None)
mae = score("8. + флаг дождя", valid["аренды"], pred)

## ── числа ──────────────────────────────────────────────────────────
print("Часов с дождём в обучении: %d (%.1f %%)"
      % ((train["дождь_идёт"] == "да").sum(),
         100 * (train["дождь_идёт"] == "да").mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(mae < 330, "ошибка должна опуститься примерно до 321")

321. Один столбец из двух значений, посчитанный из уже имеющегося,
дал пятнадцать велосипедов.

## Шаг 11. Духота: признак из произведения двух других

Тридцать градусов при влажности 30 % и тридцать при влажности 90 % —
разная погода, а модель видит их одинаково: у неё один вес на
температуру и один на влажность, и складывает она их всегда одинаково.
Чтобы модель могла различить, нужен признак, который сочетает оба:
произведение температуры и влажности.

### Задание 2.9
Добавьте в каждую часть столбец `духота` — температура, умноженная на
влажность и делённая на 100, — и включите его в числовые признаки.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
for part in (train, valid, test):
    # TODO: температура × влажность / 100
    raise NotImplementedError("замените эту строку своим решением")

ЧИСЛА = ПОГОДА + ["духота"]

подготовка = ColumnTransformer([
    ("числа", StandardScaler(), ЧИСЛА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])
model.fit(train, train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid), 0, None)
mae = score("9. + духота", valid["аренды"], pred)

## ── самопроверка ───────────────────────────────────────────────────
check(mae < 315, "ошибка должна опуститься примерно до 309")

309. Одно произведение, придуманное руками, дало двенадцать велосипедов.
Но пар у нас не одна: температура и ветер, дождь и температура,
влажность и видимость — и ещё квадраты, которые пригодятся там, где
связь гнётся (спрос падает и в мороз, и в жару).

Перебирать это руками долго. `PolynomialFeatures(degree=2,
include_bias=False)` строит сразу все квадраты и все попарные
произведения переданных ему столбцов: из семи погодных получится 35
новых. Ставим его **перед** масштабированием — произведения нужно
считать от исходных величин, а приводить к общему масштабу уже результат.

### Задание 2.10
Соберите обработку погодных столбцов из двух шагов —
`PolynomialFeatures`, затем `StandardScaler` — и обучите модель.
Столбец `духота` больше не нужен: произведение температуры на влажность
теперь строится само.

In [ ]:
## ── подготовка столбцов ────────────────────────────────────────────
# TODO: Pipeline из PolynomialFeatures(degree=2, include_bias=False) и StandardScaler
raise NotImplementedError("замените эту строку своим решением")

подготовка = ColumnTransformer([
    ("числа", погодные_произведения, ПОГОДА),
    ("категории", OneHotEncoder(handle_unknown="ignore"), КАТЕГОРИИ),
])

## ── обучение ───────────────────────────────────────────────────────
model = Pipeline([("подготовка", подготовка), ("модель", LinearRegression())])
model.fit(train, train["аренды"])

## ── качество на валидации ──────────────────────────────────────────
pred = np.clip(model.predict(valid), 0, None)
mae = score("10. произведения погоды", valid["аренды"], pred)

## ── числа ──────────────────────────────────────────────────────────
print("Столбцов после подготовки: %d" % подготовка.transform(train.head(1)).shape[1])

## ── самопроверка ───────────────────────────────────────────────────
check(mae < 290, "ошибка должна опуститься примерно до 279")

279 — лучший результат лестницы. От произведений часто ждут обратного:
столбцов становится много, модель находит среди них случайные совпадения
и начинает подгоняться под обучающую часть. Здесь этого не случилось,
и причина в том, по каким столбцам мы их строили: не по всем подряд,
а по семи погодным величинам, у которых кривизна настоящая — спрос
действительно падает и в мороз, и в жару. Так ли это на самом деле,
проверим прямо сейчас.

## Шаг 12. Нужен ли штраф

Обычное лекарство от подгонки — `Ridge`, линейная регрессия со штрафом
за большие веса. Прежде чем его применять, стоит убедиться, что болезнь
есть: сравним ошибку на обучении и на валидации и попробуем несколько
значений `alpha`.

### Задание 2.11
Посчитайте MAE итоговой модели на обучающей части (на валидации она уже
посчитана) и обучите `Ridge` с `alpha` из списка.

In [ ]:
## ── разрыв между обучением и валидацией ────────────────────────────
pred_train = np.clip(model.predict(train), 0, None)
mae_train = ...  # TODO: mean_absolute_error по train["аренды"] и pred_train

print("MAE на обучении:  %.1f" % mae_train)
print("MAE на валидации: %.1f" % mae)
print("")

## ── штраф за большие веса ──────────────────────────────────────────
print("Ridge:")
for alpha in (1, 3, 10, 30):
    ridge = Pipeline([("подготовка", clone(подготовка)), ("модель", Ridge(alpha=alpha))])
    ridge.fit(train, train["аренды"])
    p = np.clip(ridge.predict(valid), 0, None)
    print("   alpha %-3d MAE на валидации %.1f" % (alpha, mean_absolute_error(valid["аренды"], p)))

## ── самопроверка ───────────────────────────────────────────────────
check(200 < mae_train < 260, "на обучении ошибка должна быть около 227")

Штраф не помог: с ростом `alpha` ошибка только растёт. И это согласуется
с разрывом — на обучении 227, на валидации 279. Разрыв есть, но он
небольшой и объясняется тем, что валидация просто другая: осень с
высоким спросом. Лечить штрафом тут нечего.

Отрицательный результат — тоже результат: он говорит, что модель
упирается не в переобучение, а в то, чего в данных нет. Например,
в таблице нет ни одного признака про сам город — ни про ремонт дорог,
ни про метро, ни про то, что часть велосипедов сломана.

---
# Часть 3. Проверка на будущем и решение

Тест мы не трогали с самого начала. Это ноябрь 2018 — месяц, который
идёт **после** валидации, то есть настоящее будущее для нашей модели.

### Задание 3.1
Замерьте на тесте две модели: ориентир из нулевого шага (среднее
обучения) и итоговую модель с произведениями погоды.

In [ ]:
## ── тест ───────────────────────────────────────────────────────────
пусто = np.full(len(test), train["аренды"].mean())
score("тест: среднее обучения", test["аренды"], пусто, remember=False)

# TODO: прогноз обученной model на test, обрезанный снизу нулём
raise NotImplementedError("замените эту строку своим решением")
mae_test = score("тест: итоговая модель", test["аренды"], pred_test, remember=False)

## ── ошибка в долях спроса ──────────────────────────────────────────
print("")
print("В среднем за час: прогноз %.0f, факт %.0f (модель занижает на %.0f)"
      % (pred_test.mean(), test["аренды"].mean(),
         test["аренды"].mean() - pred_test.mean()))
print("")
print("Средняя ошибка в долях спроса:")
print("   валидация %.0f %%  (MAE %.0f при спросе %.0f)"
      % (100 * mae / valid["аренды"].mean(), mae, valid["аренды"].mean()))
print("   тест      %.0f %%  (MAE %.0f при спросе %.0f)"
      % (100 * mae_test / test["аренды"].mean(), mae_test, test["аренды"].mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(mae_test < 210, "на тесте ошибка должна быть около 196")

На тесте 196 против 357 у ориентира. В штуках ошибка вышла даже меньше,
чем на валидации, — но не потому, что модель стала лучше: в ноябре
холоднее и спрос ниже (719 против 1 027). В долях спроса оба числа
совпадают: 27 % и там, и там. Это и есть честная оценка — модель
ошибается примерно на четверть спроса.

И ещё одно, важное для практики: модель **занижает**. В ноябре она
обещает в среднем 576 велосипедов при фактических 719, на валидации
занижала так же — на 160. Дело не в погоде: за год прокат вырос, и
поздние месяцы активнее ранних при той же погоде. Обучение закончилось
августом, поэтому этого роста модель не знает. Занижение на будущем —
обычная плата за обучение на прошлом, и дальше мы посмотрим, что с ним
делает диспетчер.

## Шаг 13. Чего стоило бы случайное деление

Теперь вернёмся к самому началу и посчитаем цену удобного решения.
Обучим ту же самую модель на случайных 80 % всех часов и проверим
на оставшихся 20 %.

### Задание 3.2
Обучите модель на `случ_обучение` и замерьте MAE на `случ_проверка`
(эти части мы сделали в первой половине тетрадки).

In [ ]:
## ── те же столбцы для случайных частей ─────────────────────────────
for part in (случ_обучение, случ_проверка):
    part["выходной"] = np.where(
        (part["дата"].dt.dayofweek >= 5) | (part["праздник"] == "да"), "да", "нет")
    part["час_и_день"] = part["час"].astype(str) + "_" + part["выходной"]
    part["дождь_идёт"] = np.where(part["дождь"] > 0, "да", "нет")

## ── обучение ───────────────────────────────────────────────────────
случ_model = clone(model)   # та же модель, но необученная
# TODO: обучите случ_model на случ_обучение и посчитайте MAE на случ_проверка
raise NotImplementedError("замените эту строку своим решением")

## ── числа ──────────────────────────────────────────────────────────
print("Случайное деление, MAE на проверочной части: %.1f" % mae_случ)
print("Деление по времени, MAE на тесте:            %.1f" % mae_test)

## ── самопроверка ───────────────────────────────────────────────────
check(mae_случ < 250, "должно получиться около 230")

230 велосипедов — и это число выглядит убедительнее нашего. Вот только
обещать по нему нельзя ничего: модель, обученная на случайной выборке,
видела и ноябрь, и соседние часы почти каждого проверочного дня.
В отчёте так писать честно нельзя — величина «ошибка прогноза на
следующий месяц» измеряется только на месяце, которого модель не видела.

Обратите внимание, что обе цифры получены одной и той же моделью на
одних и тех же данных. Разница целиком в том, как поделили выборку.

## Шаг 14. Что диспетчер положит в машину

Прогноз в 300 велосипедов означает «в среднем 300». Если развозить
ровно столько, половину часов велосипедов будет не хватать. Диспетчеру
нужен **запас**: сколько добавить к прогнозу, чтобы хватало почти
всегда.

Считается он по промахам на валидации: смотрим, насколько модель
недооценивала спрос, и берём такой запас, чтобы покрыть 90 % часов.
`np.quantile(x, 0.9)` возвращает значение, ниже которого лежит 90 %
чисел.

### Задание 3.3
Посчитайте недооценку на валидации (факт минус прогноз), возьмите её
90-й процентиль как запас и проверьте на тесте, в какой доле часов
прогноза с запасом хватило.

In [ ]:
## ── запас по валидации ─────────────────────────────────────────────
pred_valid = np.clip(model.predict(valid), 0, None)
недооценка = valid["аренды"].values - pred_valid

запас = ...  # TODO: 90-й процентиль недооценки, np.quantile(..., 0.9)

## ── проверка на тесте ──────────────────────────────────────────────
хватило = (pred_test + запас >= test["аренды"].values).mean()
хватило_без = (pred_test >= test["аренды"].values).mean()

## ── числа ──────────────────────────────────────────────────────────
print("Запас: %.0f велосипедов сверх прогноза" % запас)
print("Хватило часов без запаса: %.0f %%" % (100 * хватило_без))
print("Хватило часов с запасом:  %.0f %%" % (100 * хватило))
print("Лишних велосипедов в среднем за час: %.0f"
      % (pred_test + запас - test["аренды"].values).mean())

## ── менее щедрые варианты ──────────────────────────────────────────
print("")
print("Если запас дорог:")
for доля in (0.7, 0.8, 0.9):
    з = np.quantile(недооценка, доля)
    print("   квантиль %.0f %%: запас %3.0f → хватило %2.0f %% часов, лишних %3.0f"
          % (100 * доля, з, 100 * (pred_test + з >= test["аренды"].values).mean(),
             (pred_test + з - test["аренды"].values).mean()))

## ── самопроверка ───────────────────────────────────────────────────
check(запас > 0, "запас — это недооценка, то есть положительное число")
check(хватило > хватило_без, "с запасом часов должно покрываться больше")

Без запаса прогноза хватало лишь в 19 % часов ноября — то самое
занижение. Запас в 598 велосипедов закрывает 96 % часов: больше
заказанных 90 %, потому что считали его по сентябрю и октябрю, где
спрос выше, и на ноябрь он щедр. Цена щедрости — 456 лишних велосипедов
на стойках в среднем за час.

Дальше это уже разговор с бизнесом, а не с моделью. Квантиль 80 % даёт
запас 433, покрытие 91 % и 291 лишний велосипед; квантиль 70 % — запас
298, покрытие 80 % и 156 лишних. Что выбрать, зависит от того, во что
прокату обходится несостоявшаяся поездка по сравнению с лишней
развозкой. Модель называет цену каждого варианта, решение принимает
прокат.

In [ ]:
## ── лестница ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.6, 4.6))
names = [tag for tag, _ in LADDER]
values = [mae for _, mae in LADDER]
colors = [GREY] + [RED if v > values[i] else BLUE
                   for i, v in enumerate(values[1:])]

ax.barh(range(len(LADDER)), values, color=colors)
ax.set_yticks(range(len(LADDER)))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel("MAE на валидации, велосипедов за час")
ax.set_title("Лестница улучшений (красным — шаги, которые сделали хуже)")
for i, value in enumerate(values):
    ax.text(value + 5, i, "%.0f" % value, va="center", fontsize=10)
plt.tight_layout()
plt.show()

---
# Итоги

| Шаг | Что сделали | MAE, велосипедов |
|---|---|---|
| 0 | одно число на все часы | 593 |
| 1 | температура | 514 |
| 2 | час как число | 431 |
| 3 | час как категория | 390 |
| 4 | вся погода | 357 |
| 5 | логарифм спроса | 434 — хуже |
| 6 | сезон категорией | 368 — хуже |
| 7 | час вместе с днём недели | 336 |
| 8 | флаг дождя | 321 |
| 9 | духота: температура × влажность | 309 |
| 10 | все произведения погодных признаков | 279 |
| — | **тест, ноябрь** | **196** против 357 у ориентира |

Что стоит унести из упражнения:

* **прогноз на будущее проверяют на будущем.** Одна и та же модель
  на случайном делении показала 230, на честном — 196 на тесте при
  27 % ошибки от спроса; первое число ничего не обещает, потому что
  95 % проверочных часов имели соседа в обучении;
* **приёмы не универсальны.** Логарифм ответа — обычное средство для
  скошенных величин — здесь ухудшил результат почти на восемьдесят
  велосипедов: ошибку мы меряем в штуках, а проверяемся на периоде
  с высоким спросом;
* **признак, значения которого в будущем новые, не работает.** Сезон
  сделал хуже, потому что осени в обучении не было; погода переносится,
  календарные метки — нет;
* **сочетание важнее набора.** Час и день недели по отдельности дали
  меньше, чем они же вместе; произведения погодных величин — больше,
  чем любой отдельный погодный столбец;
* **обучение на прошлом занижает будущее.** Прокат за год вырос,
  обучение закончилось августом — и модель занижала спрос и в сентябре
  (на 160), и в ноябре (на 142); запас закрывает именно это;
* **бизнесу нужна не метрика, а решение.** Диспетчеру мы отдали не MAE,
  а прогноз с запасом и долю часов, в которых велосипедов хватит.